# Coaxial Borehole High-Temperature Thermal Energy Storage
## Wellbore as Thermal Regenerator · Solver v1
### Coaxial Geometry · Water · CoolProp Properties · GITT Formation Model

**Physical concept:** Hot water (≤ 160 °C, cement reliability limit) is injected down the annulus
during charge, heating the near-wellbore formation. During discharge, cooler return fluid flows in
and recovers the stored heat. The wellbore acts as a *thermal regenerator*: the casing, cement and
near-formation rock form the matrix; the fluid alternately charges and extracts stored enthalpy.

**Numerical core:** Identical to the cold-TES solver (GITT formation model + transfer-matrix fluid
BVP + predictor-corrector coupling). Only three things change:
1. Boundary conditions (hot inlet on charge, cool inlet on discharge)
2. Temperature-dependent fluid properties via CoolProp
3. Performance metrics (round-trip efficiency + regenerator effectiveness instead of CCSP)

**Cement constraint:** T_cement ≤ 165 °C throughout the cycle (Class G/H Portland cement limit).


In [ ]:
!pip install CoolProp -q

import numpy as np
from scipy.special import j0, j1, y0, y1
from scipy.optimize import brentq
from scipy.linalg import expm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

try:
    from CoolProp.CoolProp import PropsSI
    COOLPROP_OK = True
    print("CoolProp available ✓")
except ImportError:
    COOLPROP_OK = False
    print("WARNING: CoolProp not found. Run: pip install CoolProp")

print("Imports OK")


## 1 · Physical Parameters

In [ ]:
# ── Geometry (legacy wellbore — same as cold-TES baseline) ────────────────
rt_i  = 0.031        # tubing inner radius [m]
rt_o  = 0.0365       # tubing outer radius [m]
rca_i = 0.0785       # casing inner radius [m]
rca_o = 0.0889       # casing outer radius [m]
rce   = 0.108        # cement outer radius = borehole wall [m]
kt    = 45.0         # tube / casing steel thermal conductivity [W/(m·K)]
kca   = 45.0         # casing steel [W/(m·K)]
kce   = 1.0          # cement [W/(m·K)]

# ── Formation ─────────────────────────────────────────────────────────────
ke      = 2.5        # formation thermal conductivity [W/(m·K)]
alpha_e = 1.263e-6   # formation thermal diffusivity [m²/s]
T_inf0  = 20.0       # surface temperature [°C]
Gg      = 0.03       # geothermal gradient [K/m]

# ── Operating conditions ──────────────────────────────────────────────────
Tin_h   = 160.0      # charge inlet temperature — hot fluid [°C]  (cement limit)
Tin_r   = 60.0       # discharge inlet temperature — return fluid [°C]
dT_op   = Tin_h - Tin_r   # = 100 K  (10× wider than cold-TES)
T_cement_limit = 165.0    # Class G/H Portland cement reliability limit [°C]

# Operating pressure: wellbore hydrostatic head >> saturation pressure at 160°C (~6 bar)
P_op = 100e5         # 100 bar [Pa] — conservative wellbore operating pressure

# ── Cycle (daily solar-thermal paradigm) ──────────────────────────────────
t_ch   = 8  * 3600   # charge duration   [s] = 8 h  (daytime solar)
t_dis  = 4  * 3600   # discharge duration [s] = 4 h  (evening demand)
Nt_ch  = 96          # charge sub-steps   (5-min each)
Nt_dis = 48          # discharge sub-steps (5-min each)
dt_ch  = t_ch  / Nt_ch
dt_dis = t_dis / Nt_dis

# ── Numerical ─────────────────────────────────────────────────────────────
NZ     = 20          # axial cells (more than cold-TES: deeper, larger gradients)
Neig   = 50          # GITT eigenvalues
Ndense = 8
Ncyc   = 5           # cycles to reach quasi-cyclic steady state

# Far-field radius
delta_thermal = np.sqrt(4 * alpha_e * t_ch)
Rfar = rce + max(2.5, 3.2 * delta_thermal)
print(f"Thermal penetration δ = {delta_thermal:.3f} m  →  Rfar = {Rfar:.3f} m")
print(f"Operating ΔT = {dT_op:.0f} K  |  Tin_h = {Tin_h}°C  |  Tin_r = {Tin_r}°C")
print(f"Cycle: {t_ch/3600:.0f} h charge  +  {t_dis/3600:.0f} h discharge")


## 2 · Temperature-Dependent Fluid Properties (CoolProp)

In [ ]:
def water_props(T_C, P_Pa=P_op):
    """
    Water thermophysical properties at temperature T_C [°C] and pressure P_Pa [Pa].
    Returns dict with rho, cp, mu, kf, Pr.
    Falls back to constant properties at 110°C if CoolProp unavailable.
    """
    if COOLPROP_OK:
        T_K = T_C + 273.15
        return {
            'rho': PropsSI('D',       'T', T_K, 'P', P_Pa, 'Water'),
            'cp':  PropsSI('C',       'T', T_K, 'P', P_Pa, 'Water'),
            'mu':  PropsSI('V',       'T', T_K, 'P', P_Pa, 'Water'),
            'kf':  PropsSI('L',       'T', T_K, 'P', P_Pa, 'Water'),
            'Pr':  PropsSI('Prandtl', 'T', T_K, 'P', P_Pa, 'Water'),
        }
    else:
        # Fallback: approximate values at 110°C
        return {'rho': 951., 'cp': 4230., 'mu': 2.52e-4, 'kf': 0.683, 'Pr': 1.56}


# ── Representative properties at key temperatures ─────────────────────────
T_ref_ch  = (Tin_h + Tin_r) / 2    # ~110°C — charge stroke mean
T_ref_dis = (Tin_r + Tin_h) / 2    # ~110°C — discharge stroke mean (same for symmetric)

props_ch  = water_props(T_ref_ch)
props_dis = water_props(T_ref_dis)

print(f"{'Property':<12}  {'Cold-TES (20°C)':>16}  {'HT-TES charge (~110°C)':>22}")
print("-" * 56)
p_cold = water_props(20.0)
for key in ['rho', 'cp', 'mu', 'kf', 'Pr']:
    print(f"{key:<12}  {p_cold[key]:>16.4g}  {props_ch[key]:>22.4g}")

print(f"\nSaturation pressure at {Tin_h}°C: "
      f"{PropsSI('P','T',Tin_h+273.15,'Q',0,'Water')/1e5:.2f} bar  "
      f"(wellbore P_op = {P_op/1e5:.0f} bar → water stays liquid ✓)" if COOLPROP_OK
      else "\nCoolProp unavailable — saturation check skipped")


## 3 · Geometry and Conductances

In [ ]:
# ── Cross-sectional areas and hydraulic diameters ─────────────────────────
Aa    = np.pi * (rca_i**2 - rt_o**2)
Atb   = np.pi * rt_i**2
Dh_a  = 2 * (rca_i - rt_o)
Dh_tb = 2 * rt_i
rlm   = (rt_o - rt_i) / np.log(rt_o / rt_i)

# ── Outer conductance: casing + cement ────────────────────────────────────
R_ca = np.log(rca_o / rca_i) / (2 * np.pi * kca)
R_ce = np.log(rce   / rca_o) / (2 * np.pi * kce)
Gext = 1.0 / (R_ca + R_ce)

# ── Cement temperature coefficients ───────────────────────────────────────
# T_cement_inner (casing outer wall) = T_a - q_per_length * R_ca
# where q_per_length = Gext * (T_a - T_bw)
# T_cao = T_a * (1 - Gext*R_ca) + T_bw * (Gext*R_ca)
alpha_cao = 1.0 - Gext * R_ca   # weight on T_a
beta_cao  = Gext * R_ca          # weight on T_bw

print(f"Aa    = {Aa:.4e} m²  |  Atb = {Atb:.4e} m²")
print(f"Dh_a  = {Dh_a:.4f} m  |  Dh_tb = {Dh_tb:.4f} m")
print(f"Gext  = {Gext:.2f} W/(m·K)")
print(f"\nCement inner wall temperature:")
print(f"  T_cao ≈ {alpha_cao:.4f}·T_a + {beta_cao:.4f}·T_bw")
print(f"  At max charge (T_a=160°C, T_bw=T_geo~50°C): "
      f"T_cao ≈ {alpha_cao*160 + beta_cao*50:.1f}°C  "
      f"({'✓ OK' if alpha_cao*160 + beta_cao*50 < T_cement_limit else '⚠ EXCEEDS LIMIT'})")


## 4 · Convective Correlations and NTU Functions

In [ ]:
def gnielinski_h(v, Dh, props, Re_clamp=3100.0):
    """Gnielinski/Petukhov Nusselt → h [W/(m²·K)] using passed fluid properties."""
    Re = props['rho'] * max(abs(v), 1e-9) * Dh / props['mu']
    Re = max(Re, Re_clamp)
    fD = (0.790 * np.log(Re) - 1.64) ** (-2)
    Nu = (fD/8) * (Re - 1000) * props['Pr'] / (
          1 + 12.7 * np.sqrt(fD/8) * (props['Pr']**(2/3) - 1))
    return Nu * props['kf'] / Dh


def compute_NTUs(va, Lbh, props):
    """
    NTUeff_at and NTUa_bw for annulus velocity va, depth Lbh,
    and fluid properties dict props.
    Returns (NTUeff_at, NTUa_bw, mdot_a).
    """
    mdot_a = props['rho'] * Aa * va
    vtb    = va * Aa / Atb
    ha     = gnielinski_h(va,  Dh_a,  props)
    htb    = gnielinski_h(vtb, Dh_tb, props)

    Gat  = 1. / (np.log(rt_o/rlm)/(2*np.pi*kt) + 1./(ha *2*np.pi*rt_o))
    Gtbt = 1. / (1./(htb*2*np.pi*rt_i)          + np.log(rlm/rt_i)/(2*np.pi*kt))

    NTUeff_at = Lbh / (mdot_a * props['cp']) / (1/Gat + 1/Gtbt)
    NTUa_bw   = Gext * Lbh / (mdot_a * props['cp'])
    return NTUeff_at, NTUa_bw, mdot_a


# ── NTU survey (compare cold-TES vs HT-TES at same velocities) ────────────
Lbh_demo = 1000.  # representative legacy well depth
print(f"NTU survey at Lbh = {Lbh_demo:.0f} m")
print(f"{'v [m/s]':>10}  {'NTUat_cold':>10}  {'NTUbw_cold':>10}  "
      f"{'NTUat_hot':>10}  {'NTUbw_hot':>10}")
print("-" * 58)
for v in [0.02, 0.05, 0.10, 0.20, 0.50]:
    at_c, bw_c, _ = compute_NTUs(v, Lbh_demo, water_props(20.))
    at_h, bw_h, _ = compute_NTUs(v, Lbh_demo, props_ch)
    print(f"{v:>10.2f}  {at_c:>10.3f}  {bw_c:>10.3f}  {at_h:>10.3f}  {bw_h:>10.3f}")


## 5 · GITT Formation Model — Eigenvalue Setup
*(Identical to cold-TES solver — physics is invariant to heat flow direction)*

In [ ]:
def gitt_char_eq(beta):
    return y1(beta*rce)*j0(beta*Rfar) - j1(beta*rce)*y0(beta*Rfar)

def compute_eigenvalues(n_eig, beta_max=500.0, n_scan=600_000):
    betas = np.linspace(1e-5, beta_max, n_scan)
    fvals = np.vectorize(gitt_char_eq)(betas)
    roots = []
    for i in range(len(betas) - 1):
        if fvals[i] * fvals[i+1] < 0:
            root = brentq(gitt_char_eq, betas[i], betas[i+1], xtol=1e-12)
            roots.append(root)
            if len(roots) == n_eig:
                break
    if len(roots) < n_eig:
        raise ValueError(f"Only {len(roots)}/{n_eig} eigenvalues found.")
    return np.array(roots)

def psi(beta, r):
    return y1(beta*rce)*j0(beta*r) - j1(beta*rce)*y0(beta*r)

def compute_norms(betas, Nr=4000):
    r = np.linspace(rce, Rfar, Nr)
    return np.array([np.trapezoid(r * psi(b,r)**2, r) for b in betas])

print("Computing GITT eigenvalues ...")
betas_arr = compute_eigenvalues(Neig)
norms_arr = compute_norms(betas_arr)
psi_rce   = psi(betas_arr, rce)
inv_wts   = psi_rce / norms_arr
forc_coeff = psi_rce / (2 * np.pi * ke * betas_arr**2)

decay_ch  = np.exp(-alpha_e * betas_arr**2 * dt_ch)
decay_dis = np.exp(-alpha_e * betas_arr**2 * dt_dis)
tau_arr   = 1.0 / (alpha_e * betas_arr**2)

print(f"  β₁  = {betas_arr[0]:.5f} m⁻¹   τ₁  = {tau_arr[0]/3600:.1f} h")
print(f"  β₅₀ = {betas_arr[-1]:.4f} m⁻¹   τ₅₀ = {tau_arr[-1]/60:.1f} min")
print("GITT setup complete ✓")


## 6 · Fluid BVP — Transfer-Matrix Shooting
**Key difference from cold-TES:** boundary conditions are generalised via `theta_in` parameters.
- Cold-TES charge: `theta_a_in = 0` (cold fluid), discharge: `theta_tb_in = 1` (warm return)
- HT-TES charge: `theta_a_in = 1` (hot fluid), discharge: `theta_tb_in = 0` (cool return)

The transfer matrix, shooting procedure, and stability logic are unchanged.


In [ ]:
def build_Phi_Psi(NTUeff_at, NTUa_bw, dz_star):
    """Matrix exponential propagator (unchanged from cold-TES)."""
    A = np.array([[-(NTUeff_at + NTUa_bw),  NTUeff_at],
                  [-NTUeff_at,               NTUeff_at]])
    Phi = expm(A * dz_star)
    Psi = np.linalg.solve(A, Phi - np.eye(2))
    return Phi, Psi


def solve_bvp_charge(NTUeff_at, NTUa_bw, theta_bw_cc, NZ, theta_a_in=0.0):
    """
    Charge BVP: annulus downward (U = +1).
    theta_a_in = 0 for cold-TES (cold fluid in);
    theta_a_in = 1 for HT-TES  (hot  fluid in).
    """
    dz = 1.0 / NZ
    Phi, Psi = build_Phi_Psi(NTUeff_at, NTUa_bw, dz)

    yH = np.zeros((NZ+1, 2))
    yP = np.zeros((NZ+1, 2))
    yH[0] = [0., 1.]
    yP[0] = [theta_a_in, 0.]    # ← generalised inlet

    for k in range(NZ):
        b_k = np.array([NTUa_bw * theta_bw_cc[k], 0.])
        yH[k+1] = Phi @ yH[k]
        yP[k+1] = Phi @ yP[k] + Psi @ b_k

    # Turnaround BC: θa(NZ) = θtb(NZ)
    s = (yP[NZ,1] - yP[NZ,0]) / (yH[NZ,0] - yH[NZ,1])
    theta = s * yH + yP
    return theta[:, 0], theta[:, 1]


def solve_bvp_discharge(NTUeff_at, NTUa_bw, theta_bw_cc, NZ, theta_tb_in=1.0):
    """
    Discharge BVP: annulus upward (U = −1), solved in reversed coordinate.
    theta_tb_in = 1 for cold-TES (warm return fluid);
    theta_tb_in = 0 for HT-TES  (cool return fluid).
    """
    dz = 1.0 / NZ
    Phi, Psi = build_Phi_Psi(NTUeff_at, NTUa_bw, dz)

    theta_bw_rev = theta_bw_cc[::-1]
    yH = np.zeros((NZ+1, 2))
    yP = np.zeros((NZ+1, 2))
    yH[0] = [1., 1.]    # turnaround: both streams equal at bottom

    for k in range(NZ):
        b_k = np.array([NTUa_bw * theta_bw_rev[k], 0.])
        yH[k+1] = Phi @ yH[k]
        yP[k+1] = Phi @ yP[k] + Psi @ b_k

    # Surface BC: θtb(top) = theta_tb_in
    s = (theta_tb_in - yP[NZ,1]) / yH[NZ,1]
    theta_rev = s * yH + yP
    return theta_rev[::-1, 0].copy(), theta_rev[::-1, 1].copy()


### 6.1 · BVP Quick-check (θbw = const = 0.5)

In [ ]:
NTUat_t, NTUbw_t, _ = compute_NTUs(0.10, 1000., props_ch)
theta_bw_test = 0.5 * np.ones(NZ)

# HT-TES BCs: theta_a_in=1 (charge), theta_tb_in=0 (discharge)
ta_ch, ttb_ch   = solve_bvp_charge(NTUat_t, NTUbw_t, theta_bw_test, NZ, theta_a_in=1.)
ta_dis, ttb_dis = solve_bvp_discharge(NTUat_t, NTUbw_t, theta_bw_test, NZ, theta_tb_in=0.)

z_star = np.linspace(0, 1, NZ+1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, ta, ttb, title in zip(axes,
        [ta_ch, ta_dis], [ttb_ch, ttb_dis],
        ['Charge — hot fluid ↓ (θa(0)=1)', 'Discharge — cool fluid ↑ (θtb(0)=0)']):
    ax.plot(z_star, ta,  label='θa (annulus)')
    ax.plot(z_star, ttb, label='θtb (tubing)', ls='--')
    ax.axhline(0.5, color='grey', lw=0.8, ls=':', label='θbw = 0.5')
    ax.set_xlabel('z*'); ax.set_ylabel('θ');  ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

assert abs(ta_ch[-1]  - ttb_ch[-1])  < 1e-9, "Charge turnaround BC violated!"
assert abs(ta_dis[-1] - ttb_dis[-1]) < 1e-9, "Discharge turnaround BC violated!"
print(f"✓ Charge   turnaround: θa(1)={ta_ch[-1]:.6f}  θtb(1)={ttb_ch[-1]:.6f}")
print(f"✓ Discharge turnaround: θa(1)={ta_dis[-1]:.6f}  θtb(1)={ttb_dis[-1]:.6f}")
print(f"  Charge   tubing return θtb(0) = {ttb_ch[0]:.4f}  "
      f"→ T = {Tin_r + ttb_ch[0]*dT_op:.1f}°C")
print(f"  Discharge annulus return θa(0) = {ta_dis[0]:.4f}  "
      f"→ T = {Tin_r + ta_dis[0]*dT_op:.1f}°C")
plt.tight_layout(); plt.show()


## 7 · Momentum Balance — Pumping Power

In [ ]:
def darcy_fD(Re):
    Re = max(Re, 3100.)
    return (0.790 * np.log(Re) - 1.64) ** (-2)

def phi_friction(va, props):
    """Combined friction coefficient using temperature-dependent props."""
    Re_a  = props['rho'] * va  * Dh_a  / props['mu']
    vtb   = va * Aa / Atb
    Re_tb = props['rho'] * vtb * Dh_tb / props['mu']
    fD_a  = darcy_fD(Re_a)
    fD_tb = darcy_fD(Re_tb)
    return fD_a + (Aa/Atb)**2 * (Dh_a/Dh_tb) * fD_tb

def pumping_power(vch, vdis, Lbh,
                  p_ch=None, p_dis=None,
                  tch=t_ch, tdis=t_dis):
    """Cycle-average pumping power [W]."""
    p_ch  = p_ch  or props_ch
    p_dis = p_dis or props_dis
    phi_ch  = phi_friction(vch,  p_ch)
    phi_dis = phi_friction(vdis, p_dis)
    return (p_ch['rho'] * Aa / (2*(tch+tdis)) * Lbh / Dh_a
            * (tch  * phi_ch  * vch**3
             + tdis * phi_dis * vdis**3))


## 8 · Regenerator Performance Metrics

| Metric | Definition | Interpretation |
|---|---|---|
| η (round-trip efficiency) | Q_dis / Q_ch | Fraction of stored heat recovered |
| ε_dis (effectiveness) | ⟨θa(0)⟩_dis | Dimensionless discharge temperature |
| Λ (reduced length) | NTU_bw | Regenerator transfer unit depth |
| Π (reduced period) | C_matrix / (ṁ cp t_cyc) | Matrix-to-fluid thermal mass ratio |


In [ ]:
def regenerator_metrics(res, Lbh, vch, props_c=None, props_d=None):
    """
    Compute regenerator effectiveness, round-trip efficiency,
    reduced length Λ and reduced period Π from solver output.
    """
    props_c = props_c or props_ch
    props_d = props_d or props_dis

    Q_ch  = res['Qch'][-1]
    Q_dis = res['Qdis'][-1]
    eta   = Q_dis / Q_ch if Q_ch > 0 else 0.0

    # Discharge effectiveness: time-average of θa(0) during discharge
    # = (Tret_dis - Tin_r) / dT_op
    eps_dis = (res['Tret_dis'][-1] - Tin_r) / dT_op

    # Reduced length Λ = NTUa_bw (charge stroke)
    _, Lambda, mdot_a = compute_NTUs(vch, Lbh, props_c)

    # Reduced period Π: formation thermal mass / fluid capacity
    # Formation mass per unit length in thermal penetration zone
    rho_cp_rock = ke / alpha_e   # volumetric heat capacity [J/(m³·K)]
    r_th  = rce + np.sqrt(4 * alpha_e * t_ch)
    C_rock_per_m = rho_cp_rock * np.pi * (r_th**2 - rce**2)
    C_fluid_total = mdot_a * props_c['cp'] * t_ch
    Pi = C_rock_per_m * Lbh / C_fluid_total

    return dict(eta=eta, eps_dis=eps_dis, Lambda=Lambda, Pi=Pi,
                Q_ch_kWh=Q_ch/3.6e6, Q_dis_kWh=Q_dis/3.6e6)

def cement_temperature(T_a_cc, Tbw_cc):
    """
    Estimate maximum cement inner-wall temperature along the borehole.
    T_cao ≈ alpha_cao * T_a + beta_cao * T_bw  (quasi-static series resistance)
    """
    return alpha_cao * T_a_cc + beta_cao * Tbw_cc


## 9 · Coupled Solver

In [ ]:
def T_inf_z(z_arr):
    """Undisturbed geothermal temperature profile T∞(z) [°C]."""
    return T_inf0 + Gg * z_arr


def run_solver_ht(Lbh, vch, vdis,
                  tch=t_ch, tdis=t_dis,
                  Nt_ch=Nt_ch, Nt_dis=Nt_dis,
                  ncyc=Ncyc, ndense=Ndense,
                  p_ch=None, p_dis=None,
                  verbose=True):
    """
    HT-TES coaxial wellbore-regenerator solver.

    Charge:    hot  fluid (θ=1) down the annulus  → heats formation
    Discharge: cool fluid (θ=0) up   the annulus  → recovers stored heat

    Parameters
    ----------
    Lbh        : borehole depth [m]
    vch, vdis  : charge / discharge annulus velocities [m/s]
    p_ch, p_dis: fluid property dicts (defaults to props_ch / props_dis)

    Returns
    -------
    dict with per-cycle scalars, Tbw profiles, metrics
    """
    p_ch  = p_ch  or props_ch
    p_dis = p_dis or props_dis

    dt_ch_loc  = tch  / Nt_ch
    dt_dis_loc = tdis / Nt_dis

    dec_ch  = np.exp(-alpha_e * betas_arr**2 * dt_ch_loc)
    dec_dis = np.exp(-alpha_e * betas_arr**2 * dt_dis_loc)

    dz_m = Lbh / NZ
    z_cc = (np.arange(NZ) + 0.5) * dz_m

    NTUat_ch,  NTUbw_ch,  mdot_ch  = compute_NTUs(vch,  Lbh, p_ch)
    NTUat_dis, NTUbw_dis, mdot_dis = compute_NTUs(vdis, Lbh, p_dis)

    if verbose:
        print(f"  NTUat / NTUbw  charge   : {NTUat_ch:.3f} / {NTUbw_ch:.3f}")
        print(f"  NTUat / NTUbw  discharge: {NTUat_dis:.3f} / {NTUbw_dis:.3f}")

    # ── State initialisation ──────────────────────────────────────────────
    U_bar = np.zeros((NZ, Neig))
    Tbw   = T_inf_z(z_cc).copy()

    hist = dict(Qdis=[], Qch=[], Tret_dis=[], Tret_ch=[],
                Tbw_end_ch=[], Tbw_end_dis=[],
                T_cement_max=[])

    def _gitt_update(U_bar, Tbw, T_a_cc, decay):
        q_bw = Gext * (T_a_cc - Tbw)
        U_bar = (U_bar * decay[np.newaxis, :]
                 + forc_coeff[np.newaxis, :]
                 * (1 - decay[np.newaxis, :])
                 * q_bw[:, np.newaxis])
        Tbw_new = T_inf_z(z_cc) + (U_bar * inv_wts[np.newaxis, :]).sum(axis=1)
        return U_bar, Tbw_new

    def _cc(theta_edge):
        return 0.5 * (theta_edge[:-1] + theta_edge[1:])

    # ── Main cycle loop ───────────────────────────────────────────────────
    for cyc in range(ncyc):
        Qch_sum = Qdis_sum = 0.0
        Tret_ch_acc = Tret_dis_acc = 0.0
        T_cement_cycle_max = 0.0

        # ── CHARGE: hot fluid (θa_in=1) flows down annulus ────────────────
        for n in range(Nt_ch):
            # Dimensionless BW temperature: θ = (T - Tin_r) / dT_op
            theta_bw = (Tbw - Tin_r) / dT_op

            # Predictor
            ta_pred, _ = solve_bvp_charge(NTUat_ch, NTUbw_ch,
                                          theta_bw, NZ, theta_a_in=1.)
            # Corrector
            theta_bw_mid = 0.5 * (theta_bw + _cc(ta_pred))
            ta, ttb = solve_bvp_charge(NTUat_ch, NTUbw_ch,
                                       theta_bw_mid, NZ, theta_a_in=1.)

            T_a_cc = Tin_r + _cc(ta) * dT_op
            U_bar, Tbw = _gitt_update(U_bar, Tbw, T_a_cc, dec_ch)

            # Cement temperature check
            T_cao = cement_temperature(T_a_cc, Tbw)
            T_cement_cycle_max = max(T_cement_cycle_max, T_cao.max())

            # Q_charge: heat delivered FROM hot fluid TO system
            # = mdot * cp * (Tin_h - T_tubing_return)
            T_tb_return = Tin_r + ttb[0] * dT_op
            Qch_sum     += mdot_ch * p_ch['cp'] * (Tin_h - T_tb_return) * dt_ch_loc
            Tret_ch_acc += T_tb_return * dt_ch_loc

        hist['Tbw_end_ch'].append(Tbw.copy())

        # ── DISCHARGE: cool fluid (θtb_in=0) flows in, recovers heat ──────
        for n in range(Nt_dis):
            theta_bw = (Tbw - Tin_r) / dT_op

            # Predictor
            ta_pred, _ = solve_bvp_discharge(NTUat_dis, NTUbw_dis,
                                             theta_bw, NZ, theta_tb_in=0.)
            # Corrector
            theta_bw_mid = 0.5 * (theta_bw + _cc(ta_pred))
            ta, ttb = solve_bvp_discharge(NTUat_dis, NTUbw_dis,
                                          theta_bw_mid, NZ, theta_tb_in=0.)

            T_a_cc = Tin_r + _cc(ta) * dT_op
            U_bar, Tbw = _gitt_update(U_bar, Tbw, T_a_cc, dec_dis)

            # Q_discharge: heat recovered BY cool fluid
            Tret_step    = Tin_r + ta[0] * dT_op    # annulus exit at surface
            Qdis_sum     += mdot_dis * p_dis['cp'] * (Tret_step - Tin_r) * dt_dis_loc
            Tret_dis_acc += Tret_step * dt_dis_loc

        hist['Tbw_end_dis'].append(Tbw.copy())
        hist['Qch'].append(Qch_sum)
        hist['Qdis'].append(Qdis_sum)
        hist['Tret_ch'].append(Tret_ch_acc  / tch)
        hist['Tret_dis'].append(Tret_dis_acc / tdis)
        hist['T_cement_max'].append(T_cement_cycle_max)

        if verbose:
            cement_warn = " ⚠ CEMENT LIMIT!" if T_cement_cycle_max > T_cement_limit else " ✓"
            print(f"  Cycle {cyc+1}: Qdis={Qdis_sum/3.6e6:.2f} kWh  "
                  f"Tret_dis={Tret_dis_acc/tdis:.1f}°C  "
                  f"Qch={Qch_sum/3.6e6:.2f} kWh  "
                  f"T_cement_max={T_cement_cycle_max:.1f}°C{cement_warn}")

    W_pump = pumping_power(vch, vdis, Lbh, p_ch, p_dis, tch, tdis)
    metrics = regenerator_metrics(hist, Lbh, vch, p_ch, p_dis)

    hist.update(dict(Tbw_final=Tbw, W_pump=W_pump,
                     mdot_ch=mdot_ch, mdot_dis=mdot_dis,
                     NTUat_ch=NTUat_ch, NTUbw_ch=NTUbw_ch,
                     NTUat_dis=NTUat_dis, NTUbw_dis=NTUbw_dis,
                     z_cc=z_cc, Lbh=Lbh, vch=vch, vdis=vdis,
                     **metrics))
    if verbose:
        print(f"  η = {metrics['eta']:.1%}  |  ε_dis = {metrics['eps_dis']:.3f}  "
              f"|  Λ = {metrics['Lambda']:.2f}  |  Π = {metrics['Pi']:.2f}")
        print(f"  W_pump = {W_pump:.1f} W")
    return hist


## 10 · Demo Run — Nominal Parameters (Lbh = 1000 m)

In [ ]:
print("=" * 60)
print("Demo: Lbh=1000 m, vch=0.05 m/s, vdis=0.20 m/s, 5 cycles")
print("=" * 60)
res = run_solver_ht(Lbh=1000., vch=0.05, vdis=0.20, ncyc=5)


### 10.1 · Results Visualisation

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

z_plot = res['z_cc']
cycles = np.arange(1, len(res['Tret_dis']) + 1)

# (a) Tbw profiles after each charge
ax1 = fig.add_subplot(gs[0,0])
for c, Tbw_c in enumerate(res['Tbw_end_ch']):
    ax1.plot(Tbw_c, z_plot, label=f'End charge {c+1}')
ax1.plot(T_inf_z(z_plot), z_plot, 'k--', lw=0.8, label='T∞(z)')
ax1.axvline(Tin_h, color='r', ls=':', lw=0.8, label=f'Tin_h={Tin_h}°C')
ax1.set_xlabel('T_bw [°C]'); ax1.set_ylabel('Depth [m]')
ax1.set_title('Borehole wall — end of charge')
ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3); ax1.invert_yaxis()

# (b) Tbw profiles after each discharge
ax2 = fig.add_subplot(gs[0,1])
for c, Tbw_d in enumerate(res['Tbw_end_dis']):
    ax2.plot(Tbw_d, z_plot, label=f'End discharge {c+1}')
ax2.plot(T_inf_z(z_plot), z_plot, 'k--', lw=0.8, label='T∞(z)')
ax2.set_xlabel('T_bw [°C]'); ax2.set_ylabel('Depth [m]')
ax2.set_title('Borehole wall — end of discharge')
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3); ax2.invert_yaxis()

# (c) Discharge return temperature per cycle
ax3 = fig.add_subplot(gs[0,2])
ax3.bar(cycles, res['Tret_dis'], color='tomato', alpha=0.8, label='T_ret_dis')
ax3.axhline(Tin_r, color='b', ls='--', lw=0.9, label=f'Tin_r = {Tin_r}°C')
ax3.axhline(Tin_h, color='r', ls='--', lw=0.9, label=f'Tin_h = {Tin_h}°C')
ax3.set_xlabel('Cycle'); ax3.set_ylabel('Avg T_ret [°C]')
ax3.set_title('Discharge return temperature'); ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3, axis='y')

# (d) Energy per cycle
ax4 = fig.add_subplot(gs[1,0])
ax4.bar(cycles - 0.2, np.array(res['Qch'])/3.6e6,  0.35, color='orange',
        alpha=0.8, label='Q_ch')
ax4.bar(cycles + 0.2, np.array(res['Qdis'])/3.6e6, 0.35, color='steelblue',
        alpha=0.8, label='Q_dis')
ax4.set_xlabel('Cycle'); ax4.set_ylabel('Energy [kWh]')
ax4.set_title('Charge / discharge energy'); ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3, axis='y')

# (e) Round-trip efficiency and effectiveness per cycle
ax5 = fig.add_subplot(gs[1,1])
eta_arr = [qd/qc if qc > 0 else 0
           for qd, qc in zip(res['Qdis'], res['Qch'])]
eps_arr = [(Tr - Tin_r)/dT_op for Tr in res['Tret_dis']]
ax5.plot(cycles, [e*100 for e in eta_arr], 'o-', color='green', label='η (%)')
ax5.plot(cycles, [e*100 for e in eps_arr], 's--', color='purple', label='ε_dis (%)')
ax5.set_xlabel('Cycle'); ax5.set_ylabel('%')
ax5.set_title('Round-trip η  &  Discharge ε')
ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

# (f) Summary panel
ax6 = fig.add_subplot(gs[1,2])
ax6.axis('off')
summary = (
    f"Lbh     = {res['Lbh']:.0f} m\n"
    f"vch     = {res['vch']:.3f} m/s\n"
    f"vdis    = {res['vdis']:.3f} m/s\n"
    f"───────────────────\n"
    f"NTUat   = {res['NTUat_ch']:.2f} / {res['NTUat_dis']:.2f}\n"
    f"Λ (NTUbw) = {res['Lambda']:.2f}\n"
    f"Π (mass)  = {res['Pi']:.2f}\n"
    f"───────────────────\n"
    f"Q_ch    = {res['Qch'][-1]/3.6e6:.2f} kWh\n"
    f"Q_dis   = {res['Qdis'][-1]/3.6e6:.2f} kWh\n"
    f"η       = {res['eta']:.1%}\n"
    f"ε_dis   = {res['eps_dis']:.3f}\n"
    f"───────────────────\n"
    f"T_cement_max = {max(res['T_cement_max']):.1f}°C\n"
    f"Cement limit = {T_cement_limit}°C  "
    f"{'✓' if max(res['T_cement_max']) < T_cement_limit else '⚠'}\n"
    f"W_pump  = {res['W_pump']:.1f} W"
)
ax6.text(0.05, 0.97, summary, transform=ax6.transAxes,
         va='top', fontsize=9, family='monospace',
         bbox=dict(boxstyle='round', facecolor='#fff4e0', alpha=0.9))
ax6.set_title('Performance summary')

plt.suptitle(
    f'Coaxial HT-TES Regenerator  |  Lbh={res["Lbh"]:.0f} m  '
    f'Tin_h={Tin_h}°C  Tin_r={Tin_r}°C  '
    f'vch={res["vch"]:.3f} m/s  vdis={res["vdis"]:.3f} m/s',
    fontsize=11, fontweight='bold')
plt.show()


## 11 · Parametric Sweep — η and ε vs (Lbh, vch)

In [ ]:
Lbh_sweep = np.array([300., 500., 750., 1000., 1250., 1500.])
vch_sweep  = np.array([0.02, 0.05, 0.10, 0.20])
vdis_fixed = 0.20

eta_grid    = np.full((len(vch_sweep), len(Lbh_sweep)), np.nan)
eps_grid    = np.full_like(eta_grid, np.nan)
Tret_grid   = np.full_like(eta_grid, np.nan)
Lambda_grid = np.full_like(eta_grid, np.nan)
cement_grid = np.full_like(eta_grid, np.nan)

print(f"{'Lbh':>6}  {'vch':>6}  {'Λ':>6}  {'Tret':>7}  {'η':>7}  {'ε_dis':>7}  {'T_cem':>7}")
print("-" * 56)
for i, vch in enumerate(vch_sweep):
    for j, Lbh in enumerate(Lbh_sweep):
        r = run_solver_ht(Lbh, vch, vdis_fixed, ncyc=Ncyc, verbose=False)
        eta_grid[i,j]    = r['eta']
        eps_grid[i,j]    = r['eps_dis']
        Tret_grid[i,j]   = r['Tret_dis'][-1]
        Lambda_grid[i,j] = r['Lambda']
        cement_grid[i,j] = max(r['T_cement_max'])
        flag = '⚠' if cement_grid[i,j] > T_cement_limit else ' '
        print(f"{Lbh:>6.0f}  {vch:>6.3f}  {r['Lambda']:>6.2f}  "
              f"{r['Tret_dis'][-1]:>7.1f}  {r['eta']:>7.1%}  "
              f"{r['eps_dis']:>7.3f}  {cement_grid[i,j]:>6.1f}°C{flag}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
titles  = ['Round-trip efficiency η [%]', 'Discharge effectiveness ε_dis',
           'Discharge return T [°C]',     'Max cement temperature [°C]']
grids   = [eta_grid*100, eps_grid, Tret_grid, cement_grid]
cmaps   = ['viridis', 'viridis', 'RdYlGn', 'RdYlGn_r']

for ax, data, title, cmap in zip(axes.flat, grids, titles, cmaps):
    im = ax.pcolormesh(Lbh_sweep, vch_sweep*100, data,
                       shading='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('Lbh [m]'); ax.set_ylabel('vch [cm/s]')
    ax.set_title(title)
    for i in range(len(vch_sweep)):
        for j in range(len(Lbh_sweep)):
            ax.text(Lbh_sweep[j], vch_sweep[i]*100,
                    f'{data[i,j]:.1f}',
                    ha='center', va='center', fontsize=7,
                    color='white', fontweight='bold')

# Overlay cement limit contour
cs = axes[1,1].contour(Lbh_sweep, vch_sweep*100, cement_grid,
                        levels=[T_cement_limit], colors='red', linewidths=2)
axes[1,1].clabel(cs, fmt=f'{T_cement_limit}°C limit', fontsize=8)

plt.suptitle(
    f'HT-TES Parametric Sweep  |  vdis={vdis_fixed} m/s  |  {Ncyc} cycles  '
    f'|  Tin_h={Tin_h}°C  Tin_r={Tin_r}°C',
    fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()
